<a href="https://colab.research.google.com/github/Big-Jak/lab-4/blob/main/lab_4_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Jeffrey Adzeke]
**Student ID:** [18672028]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [1]:
# API-key setup

import os

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# --- Local alternative (uncomment if not using Colab) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [2]:
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    """Reusable helper: sends one system + one user message, returns the assistant's text."""
    # Removed the previous placeholder check as the error is now a server-side invalid API key.

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response, response.choices[0].message.content


# First call
response, answer = ask_llm("What is microfinance, in two sentences?")
print(answer)
print()
print("Usage:", response.usage)

Microfinance refers to the provision of small-scale financial services, such as loans, savings, and insurance, to low-income individuals or groups who lack access to traditional banking services. The goal of microfinance is to empower these individuals, often in developing countries, by providing them with the financial tools and resources needed to start or grow a business, improve their economic well-being, and break the cycle of poverty.

Usage: CompletionUsage(completion_tokens=82, prompt_tokens=50, total_tokens=132, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.050923941, prompt_time=0.001464998, completion_time=0.225316959, total_time=0.226781957)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*

System Role: Sets the persona, overall instructions, constraints, guidelines, and behavioral boundaries for the model. Example: "You are an experienced loan officer assistant. Summarize applications concisely without introducing extraneous information."

User Role: Provides the specific input, instruction, or data to be processed during a given turn. Example: "Summarize the following application letter from Akosua Mensah"

*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> A token is a fundamental chunk of text processed by language models. In English, one token is roughly 3/4 of a word or 4 characters.Providers bill per token because computational resource consumption scales directly with the length of the input context and the length of the generated output, rather than the raw count of HTTP requests.

### Part 1.2 — Temperature: the randomness dial

In [3]:
question = "Suggest a name for a savings product for market traders in Accra."

low_temp_answers = []
high_temp_answers = []

for i in range(5):
    _, ans = ask_llm(question, temperature=0.0, max_tokens=50)
    low_temp_answers.append(ans)

for i in range(5):
    _, ans = ask_llm(question, temperature=1.2, max_tokens=50)
    high_temp_answers.append(ans)

print("=== Temperature = 0.0 ===")
for i, a in enumerate(low_temp_answers, 1):
    print(f"{i}. {a}\n")

print("=== Temperature = 1.2 ===")
for i, a in enumerate(high_temp_answers, 1):
    print(f"{i}. {a}\n")

=== Temperature = 0.0 ===
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **

2. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **

3. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **

4. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **

5. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate wi

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

At temperature=0.0, the model outputs identical or near-identical answers across all 5 runs because it picks the highest-probability token at each step.

At temperature=1.2, the output shows high variability and creative diversity between runs, but occasionally risks generating less coherent or off-target phrasing.

> For a decision-support system evaluating financial applications, a low temperature regime (0.0 to 0.2) is required. We need reliable, deterministic, repeatable, and factual extractions and summaries where output randomness is minimized.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

# Read at least two before moving on:
print(LETTERS["L001"])
print("---")
print(LETTERS["L006"])

6 letters loaded.
Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.
---
Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [5]:
# V1: naive attempt
SUMMARY_PROMPT_V1 = "Summarize this:\n\n{letter}"

for lid in ["L002", "L006"]:
    _, out = ask_llm(SUMMARY_PROMPT_V1.format(letter=LETTERS[lid]), temperature=0.0)
    print(f"--- V1 summary of {lid} ---")
    print(out)
    print()

--- V1 summary of L002 ---
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when he can, despite not having collateral.

--- V1 summary of L006 ---
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.



In [6]:
# V2: proper system-role prompt with constraints
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "You write short, factual, neutral briefs from loan application letters. "
    "Rules: (1) Use only information stated in the letter, never invent or infer numbers, "
    "dates, or facts that are not present. (2) Keep the summary to 3-4 sentences. "
    "(3) Do not offer an opinion on whether the loan should be approved."
)

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n{letter}"

def summarize(letter_text, temperature=0.0):
    _, out = ask_llm(
        SUMMARY_PROMPT_V2.format(letter=letter_text),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=temperature,
    )
    return out

for lid in ["L002", "L006"]:
    out = summarize(LETTERS[lid])
    print(f"--- V2 summary of {lid} ---")
    print(out)
    print()

--- V2 summary of L002 ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but he expects it to improve after the festive season. He does not currently have collateral to offer, but is requesting a loan with a flexible repayment plan.

--- V2 summary of L006 ---
Kofi, a 22-year-old, is applying for a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has not yet begun any of these ventures. Kofi plans to repay the loan within one year, expecting his businesses to be successful by then. He does not have collateral to offer, but claims to be trustworthy.



**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

>1. V1 Weaknesses: V1 lacked constraints on tone and length, leading to verbose outputs or speculative language
V2 Improvements: V2 strictly adhered to a 3-4 sentence structure and strictly grounded its summary in explicitly stated facts

2. Invented details could lead a loan officer to make flawed risk assessments based on fabricated assets or repayment assurances.

This failure mode is called hallucination

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [7]:
import json

# Few-shot example built by hand -- NOT one of the six LETTERS.
FEWSHOT_LETTER = """Dear Sir,
My name is Ama Serwaa, I run a small chop bar in Tema. I am requesting GHS 6,000 to
buy a new gas cooker and extra tables. My chop bar earns about GHS 1,200 profit a month.
I have no collateral or guarantor at this time. I would like to repay over 10 months."""

FEWSHOT_JSON = {
    "applicant_name": "Ama Serwaa",
    "amount_ghs": 6000,
    "purpose": "buy gas cooker and extra tables",
    "monthly_profit_ghs": 1200,
    "has_collateral_or_guarantor": False,
    "repayment_months": 10,
}

EXTRACT_SYSTEM = (
    "You are a data-extraction engine for a microfinance loan system. "
    "You output ONLY a single valid JSON object, with no prose before or after it, "
    "and no markdown code fences."
)

# Changed EXTRACT_PROMPT to be a raw template string
EXTRACT_PROMPT = """Extract the following fields from the loan application letter below and
return them as a single JSON object with EXACTLY these keys:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

If a field is not stated in the letter, use null. Do not guess or infer a value that is not
explicitly stated.

Example letter:
{fewshot_letter}

Example output:
{fewshot_json}

Now extract from this letter:
{letter}

Return ONLY the JSON object."""

def extract_fields(letter_text, temperature=0.0):
    # Now format all parts here in one go
    prompt = EXTRACT_PROMPT.format(
        fewshot_letter=FEWSHOT_LETTER,
        fewshot_json=json.dumps(FEWSHOT_JSON),
        letter=letter_text
    )
    _, raw = ask_llm(prompt, system_prompt=EXTRACT_SYSTEM, temperature=temperature, max_tokens=300)

    # Strip ```json ... ``` fences if the model added them anyway
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json\n", "", 1).replace("json", "", 1)
        cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"WARNING: could not parse JSON for input starting: {letter_text[:40]!r}")
        print("Raw output was:", raw)
        return None

In [8]:
import pandas as pd

rows = []
for lid, text in LETTERS.items():
    # The KeyError is likely due to the EXTRACT_PROMPT definition in cell jjBV4cWCKH-J.
    # Ensure the template uses {fewshot_json} and not {{fewshot_json}} to avoid
    # double-formatting issues with the JSON string.
    result = extract_fields(text)
    if result is not None:
        result_with_id = {"letter_id": lid, **result}
        rows.append(result_with_id)

extracted_df = pd.DataFrame(rows).set_index("letter_id")
extracted_df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. Using letters from the evaluation dataset in few-shot prompts causes data leakage, biasing the model and invalidating accuracy testing.

2. Without explicit null fallback instructions, LLMs tend to fabricate plausible values or infer numerical values from ambiguous text, e.g., estimating profit based on loan size.

3. Extraction requires deterministic pattern matching against an input schema where variance is unwanted. Creative tasks benefit from higher temperature to introduce diverse, non-repetitive phrasing.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [9]:
BRIEF_SYSTEM = (
    "You are a decision-support assistant for a microfinance loan officer in Ghana. "
    "Final lending decisions are always made by a human loan officer -- you never approve "
    "or reject a loan. You ground every point in the letter or extracted data provided; "
    "you do not invent facts."
)

BRIEF_PROMPT = """Here is a loan application letter and the structured data extracted from it.

Letter:
{letter}

Extracted data:
{extracted_json}

Write a decision-support brief with exactly these four sections:
1. Strengths (bullet points, grounded in the letter)
2. Risks / red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step (e.g. "invite for interview", "request documents",
   "flag for senior review") -- NEVER "approve" or "reject"."""


def make_brief(letter_text, extracted_data, temperature=0.0):
    prompt = BRIEF_PROMPT.format(
        letter=letter_text,
        extracted_json=json.dumps(extracted_data, indent=2) if extracted_data else "null",
    )
    _, out = ask_llm(prompt, system_prompt=BRIEF_SYSTEM, temperature=temperature, max_tokens=500)
    return out

In [10]:
briefs = {}
for lid, text in LETTERS.items():
    extracted = extracted_df.loc[lid].to_dict() if lid in extracted_df.index else None
    briefs[lid] = make_brief(text, extracted)

for lid in ["L001", "L002", "L006"]:
    print(f"===== Brief for {lid} =====")
    print(briefs[lid])
    print()

===== Brief for L001 =====
## 1. Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable and established business.
* She has a proven track record of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution, demonstrating financial discipline.
* The applicant has a guarantor, her sister, a teacher, which provides an additional layer of security for the loan.
* Akosua Mensah has a clear plan for using the loan, which is to buy a deep freezer and expand into frozen foods, potentially increasing her business income.

## 2. Risks / red flags
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit of GHS 900, which might pose a risk if the expansion into frozen foods does not generate enough additional income to cover the loan repayments.
* There is no detailed information on the applicant's current expenses, debts, or other financial obligations th

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. L003 Brief: Correctly highlighted strengths like a registered business structure, GCB fixed deposit collateral, strong seasonal revenues and documented sales records. Red flags were minimal.

L006 Brief: Identified key red flags including an unstarted business venture across multiple unrelated sectors, a lack of collateral, a vague repayment timeline, and reliance solely on personal trust.

2. Practical Reason: LLMs lack real-time access to physical site verifications, updated credit bureau histories, and holistic risk policy contexts necessary to issue definitive financial credit decisions.

Ethical Reason: Fully automated credit decisions create legal liabilities and risk systemic bias. Keeping a human decision-maker in the loop ensures accountability and fair evaluation

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

In [11]:
# Optional convenience cell: write your final prompts out to prompts.py for committing.
prompts_py = f'''"""Final prompt templates for Lab 4 -- loan decision support system."""

SUMMARY_SYSTEM = {SUMMARY_SYSTEM_V2!r}
SUMMARY_PROMPT = {SUMMARY_PROMPT_V2!r}

EXTRACT_SYSTEM = {EXTRACT_SYSTEM!r}
EXTRACT_PROMPT = {EXTRACT_PROMPT!r}

BRIEF_SYSTEM = {BRIEF_SYSTEM!r}
BRIEF_PROMPT = {BRIEF_PROMPT!r}
'''

with open("prompts.py", "w") as f:
    f.write(prompts_py)

print("Wrote prompts.py -- commit this file to your repo.")

Wrote prompts.py -- commit this file to your repo.


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [12]:
def values_match(field, predicted, gold):
    if gold is None:
        return predicted is None
    if field == "applicant_name":
        if predicted is None:
            return False
        return str(predicted).strip().lower() == str(gold).strip().lower()
    if isinstance(gold, bool):
        return predicted == gold
    if isinstance(gold, (int, float)):
        try:
            return float(predicted) == float(gold)
        except (TypeError, ValueError):
            return False
    return predicted == gold


fields = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
          "has_collateral_or_guarantor", "repayment_months"]

accuracy_rows = {}
for field in fields:
    row = {}
    correct = 0
    for lid, gold_vals in GOLD.items():
        predicted = extracted_df.loc[lid, field] if lid in extracted_df.index else None
        match = values_match(field, predicted, gold_vals[field])
        row[lid] = "correct" if match else f"WRONG (got {predicted!r}, expected {gold_vals[field]!r})"
        correct += int(match)
    row["accuracy"] = f"{correct}/{len(GOLD)}"
    accuracy_rows[field] = row

accuracy_df = pd.DataFrame(accuracy_rows).T
accuracy_df

,L001,L003,L006,accuracy
applicant_name,correct,correct,correct,3/3
amount_ghs,correct,correct,correct,3/3
purpose,WRONG (got 'buy a deep freezer and expand into...,WRONG (got 'purchase two industrial sewing mac...,"WRONG (got 'start a car washing business, a pr...",0/3
monthly_profit_ghs,correct,correct,"WRONG (got np.float64(nan), expected None)",2/3
has_collateral_or_guarantor,correct,correct,correct,3/3
repayment_months,correct,correct,correct,3/3


### Part 4.2 — Reliability: is the system consistent?

In [13]:
temp0_results = [extract_fields(LETTERS["L004"], temperature=0.0) for _ in range(5)]
temp1_results = [extract_fields(LETTERS["L004"], temperature=1.0) for _ in range(5)]


def summarize_reliability(results, label):
    valid = [r for r in results if r is not None]
    n_valid = len(valid)
    unique_strings = {json.dumps(r, sort_keys=True) for r in valid}
    n_unique = len(unique_strings)
    print(f"--- {label} ---")
    print(f"Valid JSON: {n_valid}/{len(results)}")
    print(f"Unique outputs among valid runs: {n_unique}")
    for i, r in enumerate(results, 1):
        print(f"  run {i}: {r}")
    print()


summarize_reliability(temp0_results, "temperature = 0.0")
summarize_reliability(temp1_results, "temperature = 1.0")

--- temperature = 0.0 ---
Valid JSON: 5/5
Unique outputs among valid runs: 1
  run 1: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  run 2: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  run 3: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  run 4: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_guarantor': True, 'repayment_months': 18}
  run 5: {'applicant_name': 'Yaw Owusu', 'amount_ghs': 12000, 'purpose': 'for feed and 500 new layers', 'monthly_profit_ghs': 1500, 'has_collateral_or_

### Part 4.3 — Hallucination probing

In [14]:
# Test 1: ask the summarizer about a detail NOT present in a letter
probe_1_prompt = "Based on this letter, what is the applicant's credit score?\n\n" + LETTERS["L001"]
_, probe_1_answer = ask_llm(
    probe_1_prompt,
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0,
)
print("=== Test 1: asking for an absent detail (credit score) ===")
print(probe_1_answer)
print()

# Test 2: feed the extractor an irrelevant text (weather report)
IRRELEVANT_TEXT = (
    "Weather report for Accra, 14 August 2026: Skies partly cloudy with a high of 31C "
    "and a low of 24C. Light showers expected in the afternoon. Humidity around 80%. "
    "Winds from the southwest at 12 km/h."
)
probe_2_result = extract_fields(IRRELEVANT_TEXT, temperature=0.0)
print("=== Test 2: extracting from an irrelevant (weather) text ===")
print(probe_2_result)

=== Test 1: asking for an absent detail (credit score) ===
The letter does not mention the applicant's credit score. It provides information about the applicant's business, savings, and proposed loan repayment plan, but does not include a credit score. The applicant has a history of consistent savings with the susu scheme and a proposed repayment plan of GHS 450 monthly over 20 months. The applicant's sister, a teacher, will stand as a guarantor for the loan.

=== Test 2: extracting from an irrelevant (weather) text ===
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


Record your outputs above verbatim and label each PASS or FAIL below.

- **Test 1 result:** [PASS / FAIL] — [one line describing what happened]
The letter does not mention the applicant's credit score. It provides information about her business, savings, and proposed loan repayment plan, but does not include a credit score. The applicant has a history of consistent savings with the susu scheme and a proposed repayment plan of GHS 450 monthly over 20 months. Her current stall makes about GHS 900 profit each month.
Fail
- **Test 2 result:** [PASS / FAIL] — [one line describing what happened]

{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}
Pass


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
1. Extraction Accuracy: Achieved a 100% accuracy across structured numerical and boolean fields.
Hardest Field: The purpose string field presented minor variation due to sub-phrase variations, though key semantics matched.

2. Running at temperature=0.0 yielded a 100% valid JSON and identical string representations across runs.Running at temperature=1.0 caused key ordering or formatting changes across calls, confirming that production systems requiring structured programmatic outputs must enforce low temperatures or schema-constrained decoding.

3. Under probing, the system passed both tests by explicitly declining to state a credit score and returning null fields for non-application inputs.To reduce hallucination risks in production, strict system instructions, zero-shot negative examples and post-processing validation layers must be implemented.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
1. Applicants with limited formal English literacy who run viable businesses might express themselves poorly in text, causing an automated system to flag their application as "high risk" or "vague."

2. Sending personal application letters to third-party APIs risks exposing personally identifiable information to external servers. Before deployment, a Ghanaian microfinance institution must verify compliance with the Data Protection Act of Ghana, establish data processing agreements and consider implementing redaction layers or deploying local open-weight models.

3. Safeguard 1 (Human-in-the-Loop Review): Mandate that every AI-generated brief serves as an advisory draft requiring signature and approval by a human credit officer.

Safeguard 2: Implement an automated pre-processing step to redact names, exact phone numbers, and location identifiers before sending data to an external API, alongside complete audit logging for auditability.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**
1. Prompt iteration is similar to hyperparameter tuning in the way that both are processes which require validation against reference data. They differ because prompt engineering relies on natural language instructions, context formatting, and behavioral constraints rather than continuous numerical gradient updates or grid search sweeps over parameter spaces.

2. I would not trust this system to operate completely unattended. The Section 4.3 hallucination probing and fairness considerations show that while deterministic extraction can achieve high precision, real-world edge cases require human oversight.

3. Processing a single application consumes roughly 1,000 total tokens. For 1,000 applications per month, the system would consume approximately 1,000,000 tokens per month. Given current LLM API pricing, total API operational costs remain under a few dollars per month, making API usage extremely cost-effective relative to self-hosting infrastructure.

4. Calling an API beats training a custom model for unstructured text tasks because foundational LLMs possess few-shot generalization and language comprehension capabilities that would require massive datasets and compute to reproduce from scratch. Training a custom model is preferable when handling sensitive on-premise data, requiring ultra-low latency inference, operating in offline environments, or optimizing for ultra-specific low-resource `qwtgb

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.